# 02 - Data Preparation (v2)

Cleaning checks and the dataset-size decision (sample vs. keep full 800K). No train/validation/test split is saved here: same pattern as `legacy_v1`, each modelling notebook (`04`, `05`, `06`) re-derives the identical split inline from `dataset_pe_v2_full.csv`, using the same fixed `RANDOM_STATE`, rather than persisting extra split CSVs to disk.

In [1]:
import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv("../data/dataset_pe_v2_full.csv")
feature_cols = [c for c in df.columns if c not in ("Name", "Malware", "OriginalSplit")]
print("shape:", df.shape)

shape: (800000, 23)


## Cleaning checks

In [2]:
before = len(df)
dupes = df["Name"].duplicated().sum()
print(f"duplicate SHA-256 hashes: {dupes} (would drop {dupes} rows if any)")

duplicate SHA-256 hashes: 0 (would drop 0 rows if any)


In [3]:
zero_var = [c for c in feature_cols if df[c].std() == 0]
print("zero-variance feature columns:", zero_var if zero_var else "none")

zero-variance feature columns: none


In [4]:
bad = df[feature_cols].isin([np.inf, -np.inf]).sum().sum() + df[feature_cols].isnull().sum().sum()
print(f"infinite or NaN feature values: {int(bad)}")

infinite or NaN feature values: 0


No rows need to be dropped: 0 duplicate hashes, 0 zero-variance columns, 0 invalid values. `dataset_pe_v2_full.csv` is used as-is by every downstream notebook, no separate "cleaned" copy needed (unlike v1, where cleaning on `dataset_malwares.csv` did drop rows and a `dataset_cleaned.csv` was saved for `03_eda.ipynb`).

## Dataset size decision: keep the full 800,000 rows, no downsampling

This is the key evaluated decision Darrius asked for explicitly (the original v1 project used only ~19,600 rows, and the working hypothesis for its ~90% real-world false-positive rate was that this was both too small and not representative of ordinary software). Sampling this new, much larger dataset down again without a real reason would risk reintroducing the same problem, so this decision is made and justified here rather than baked silently into the CSV build step.

**Reasoning, based on industry/published practice, not preference:**

1. **The EMBER paper's own baseline (Anderson & Roth, 2018, arXiv:1804.04637) trains on the complete labeled training corpus, not a subsample**, and reports ROC AUC 0.999 with well under 1% false positives at over 98% detection. Downsampling was never part of how that benchmark number was produced, so shrinking this project's dataset for convenience would be a real, avoidable step backward from the reference this project is modelling itself on.
2. **Tree ensembles (Random Forest, XGBoost, LightGBM) scale close to linearly with row count** and routinely train on hundreds of thousands to millions of rows within minutes on ordinary laptop hardware, this is a well-established property of these algorithms (unlike, say, kernel SVMs or large neural nets, which is why EMBER's own baseline uses a gradient-boosted tree model, LightGBM, in the first place). 800,000 rows x 20 features is not a genuine compute obstacle for this model family.
3. **More real, labeled data directly reduces the risk of the exact problem this rebuild exists to fix**: a benign class that doesn't structurally resemble typical real-world software. Shrinking the dataset only makes sense when there's an actual compute or time constraint forcing the trade-off; there isn't one here.

**Where a smaller subsample is still legitimately used later:** `06_evaluation.ipynb`'s SHAP explainability step uses a random subsample (a few thousand rows) purely because SHAP summary/beeswarm plots don't get more informative or readable past a certain point and full-scale plotting is wasted compute, not because the training data itself is being reduced. That is a plotting-practicality decision, not a data-quantity decision, and it happens after training, not before. Separately, `04`/`05`/`06`'s `GridSearchCV`/training step downsamples the ~600,000-row training pool to 150,000 rows (75,000 per class) purely for tuning-time compute reasons on ordinary laptop hardware; this is a disclosed, documented tradeoff (see the note directly above the split cell below), and it does not touch the held-out EMBER `test` split, so final reported test metrics in `06` remain on the full, standard 200,000-row benchmark split.

## Train / validation / test split (reference only, not persisted)

`04`, `05`, and `06` each independently reload `dataset_pe_v2_full.csv` and derive the same split shown below, using this fixed `RANDOM_STATE`, the same reproducibility approach `legacy_v1` uses (no split files saved to disk, every notebook can regenerate the exact same rows on its own). EMBER's own official `train`/`test` boundary (`OriginalSplit` column) is kept as the outer split (matches the published benchmark, avoids introducing leakage via an in-house re-split); an 85/15 stratified split of the `train` rows produces the validation set. Shown here once for reference.

**Compute-constrained downsample of the training pool (2026-07, added after real local testing).** `GridSearchCV`'s exhaustive search over the full ~600,000-row training pool proved too slow on typical laptop hardware (each of the three grid searches trains many separate models across cross-validation folds). Downsamples `train_pool` to 75,000 rows per class (150,000 total, still roughly 7-8x larger than v1's ~19,600-row dataset) before the 85/15 train/validation split, using a fixed `RANDOM_STATE` so the reduction is reproducible. **The held-out `test` set (EMBER's own official 200,000-row split) is NOT reduced**: `06_evaluation.ipynb`'s final reported numbers still come from the complete, standard benchmark split, only the expensive tuning step works on a smaller pool. This is a disclosed, documented tradeoff for real hardware constraints, not a silent change.

In [5]:
train_pool = df[df["OriginalSplit"] == "train"].reset_index(drop=True)
test_df = df[df["OriginalSplit"] == "test"].reset_index(drop=True)

TRAIN_POOL_SAMPLE_PER_CLASS = 75000
train_pool = pd.concat([
    g.sample(n=min(TRAIN_POOL_SAMPLE_PER_CLASS, len(g)), random_state=RANDOM_STATE)
    for _, g in train_pool.groupby("Malware")
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print("downsampled train_pool:", train_pool.shape)

train_df = pd.concat([
    g.sample(frac=0.85, random_state=RANDOM_STATE)
    for _, g in train_pool.groupby("Malware")
]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
val_df = train_pool.drop(
    pd.concat([g.sample(frac=0.85, random_state=RANDOM_STATE) for _, g in train_pool.groupby("Malware")]).index
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pct = d["Malware"].value_counts(normalize=True).sort_index() * 100
    print(f"{name}: n={len(d)}  benign={pct[0]:.2f}%  malicious={pct[1]:.2f}%")

downsampled train_pool: (150000, 23)
train: n=127500  benign=50.00%  malicious=50.00%
val: n=22500  benign=50.00%  malicious=50.00%
test: n=200000  benign=50.00%  malicious=50.00%


## Summary

No rows dropped (0 duplicates, 0 zero-variance, 0 invalid values), full 800,000-row dataset retained (decision justified above). Reference split (training pool downsampled to 150,000 rows for tuning compute, see note above): ~127,500 train / ~22,500 val / 200,000 held-out test, stratified by class, test set is EMBER's own official split, untouched. Not saved to disk, `04`/`05`/`06` each reproduce it inline with the same `RANDOM_STATE`. Next: `03_eda.ipynb` explores feature distributions and correlations.